# Project 1 — Exploratory Data Analysis

## Supply Chain Inventory & Demand Planning

This notebook explores product demand, category demand, warehouse demand,
stockouts, inventory levels, and monthly demand patterns.

The analysis identifies:
- demand concentration
- high-demand products and categories
- warehouse-level demand and stockout exposure
- monthly demand patterns
- products requiring inventory-management attention

All calculations use the cleaned datasets in `data/processed/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Locate project root robustly
cwd = Path.cwd()
BASE_DIR = cwd if (cwd / "data").exists() else cwd.parent
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

suppliers = pd.read_csv(PROCESSED_DATA_DIR / "suppliers_clean.csv")
products = pd.read_csv(PROCESSED_DATA_DIR / "products_clean.csv")
inventory = pd.read_csv(PROCESSED_DATA_DIR / "inventory_clean.csv")
sales = pd.read_csv(PROCESSED_DATA_DIR / "sales_clean.csv")
purchase_orders = pd.read_csv(PROCESSED_DATA_DIR / "purchase_orders_clean.csv")

sales["order_date"] = pd.to_datetime(sales["order_date"])
inventory["date"] = pd.to_datetime(inventory["date"])
purchase_orders["order_date"] = pd.to_datetime(purchase_orders["order_date"])
purchase_orders["delivery_date"] = pd.to_datetime(purchase_orders["delivery_date"])

print("Datasets loaded successfully.")
print("Suppliers:", suppliers.shape)
print("Products:", products.shape)
print("Inventory:", inventory.shape)
print("Sales:", sales.shape)
print("Purchase Orders:", purchase_orders.shape)


## 1. Overall Demand

In [ ]:
total_ordered = sales["quantity_ordered"].sum()
total_fulfilled = sales["quantity_fulfilled"].sum()
total_unfulfilled = total_ordered - total_fulfilled
fill_rate = total_fulfilled / total_ordered * 100

overall_summary = pd.DataFrame({
    "Metric": [
        "Total demand ordered",
        "Total quantity fulfilled",
        "Unfulfilled quantity",
        "Fill rate (%)",
        "Unfulfilled demand rate (%)"
    ],
    "Value": [
        total_ordered,
        total_fulfilled,
        total_unfulfilled,
        fill_rate,
        100 - fill_rate
    ]
})

overall_summary


## 2. Product Demand Analysis

In [ ]:
product_demand = (
    sales.groupby("product_id", as_index=False)
    .agg(
        total_demand=("quantity_ordered", "sum"),
        total_fulfilled=("quantity_fulfilled", "sum")
    )
    .merge(
        products[
            [
                "product_id",
                "product_name",
                "category",
                "warehouse_id",
                "unit_cost",
                "supplier_id"
            ]
        ],
        on="product_id",
        how="left"
    )
)

product_demand["demand_share_pct"] = (
    product_demand["total_demand"]
    / product_demand["total_demand"].sum()
    * 100
)

product_demand = product_demand.sort_values(
    "total_demand",
    ascending=False
)

product_demand["cumulative_demand_share_pct"] = (
    product_demand["demand_share_pct"].cumsum()
)

product_demand


In [ ]:
top3_demand_share = product_demand.head(3)["demand_share_pct"].sum()
top10_demand_share = product_demand.head(10)["demand_share_pct"].sum()

print(f"Top 3 products account for {top3_demand_share:.2f}% of total demand.")
print(f"Top 10 products account for {top10_demand_share:.2f}% of total demand.")


In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    product_demand["product_name"],
    product_demand["total_demand"],
    color=plt.cm.tab20(np.linspace(0, 1, len(product_demand)))
)

plt.title("Demand by Product")
plt.xlabel("Product")
plt.ylabel("Units Ordered")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()


## 3. Category Demand Analysis

In [ ]:
category_demand = (
    product_demand
    .groupby("category", as_index=False)
    .agg(total_demand=("total_demand", "sum"))
    .sort_values("total_demand", ascending=False)
)

category_demand["demand_share_pct"] = (
    category_demand["total_demand"]
    / category_demand["total_demand"].sum()
    * 100
)

category_demand


In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    category_demand["category"],
    category_demand["total_demand"],
    color=plt.cm.Set2(np.linspace(0, 1, len(category_demand)))
)

plt.title("Demand by Product Category")
plt.xlabel("Category")
plt.ylabel("Units Ordered")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 4. Warehouse Demand and Stockout Analysis

In [ ]:
warehouse_demand = (
    sales
    .groupby("warehouse_id", as_index=False)
    .agg(
        total_demand=("quantity_ordered", "sum"),
        total_fulfilled=("quantity_fulfilled", "sum")
    )
)

warehouse_demand["unfulfilled_units"] = (
    warehouse_demand["total_demand"]
    - warehouse_demand["total_fulfilled"]
)

warehouse_demand["stockout_rate_pct"] = (
    warehouse_demand["unfulfilled_units"]
    / warehouse_demand["total_demand"]
    * 100
)

warehouse_demand["demand_share_pct"] = (
    warehouse_demand["total_demand"]
    / warehouse_demand["total_demand"].sum()
    * 100
)

warehouse_demand.sort_values(
    "total_demand",
    ascending=False
)


In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    warehouse_demand["warehouse_id"],
    warehouse_demand["stockout_rate_pct"],
    color=plt.cm.tab10(np.arange(len(warehouse_demand)))
)

plt.title("Stockout / Unfulfilled Demand Rate by Warehouse")
plt.xlabel("Warehouse")
plt.ylabel("Unfulfilled Demand (%)")
plt.tight_layout()
plt.show()


## 5. Monthly Demand Trend

In [ ]:
monthly_demand = (
    sales
    .assign(
        month=sales["order_date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("month", as_index=False)
    .agg(
        demand=("quantity_ordered", "sum"),
        fulfilled=("quantity_fulfilled", "sum")
    )
)

monthly_demand["unfulfilled"] = (
    monthly_demand["demand"]
    - monthly_demand["fulfilled"]
)

monthly_demand["stockout_rate_pct"] = (
    monthly_demand["unfulfilled"]
    / monthly_demand["demand"]
    * 100
)

monthly_demand


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    monthly_demand["month"],
    monthly_demand["demand"],
    marker="o",
    linewidth=2,
    label="Demand"
)

plt.plot(
    monthly_demand["month"],
    monthly_demand["fulfilled"],
    marker="o",
    linewidth=2,
    label="Fulfilled"
)

plt.title("Monthly Demand vs Fulfilled Quantity")
plt.xlabel("Month")
plt.ylabel("Units")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    monthly_demand["month"].dt.strftime("%b"),
    monthly_demand["stockout_rate_pct"],
    color=plt.cm.viridis(
        np.linspace(0.1, 0.9, len(monthly_demand))
    )
)

plt.title("Monthly Unfulfilled Demand Rate")
plt.xlabel("Month")
plt.ylabel("Unfulfilled Demand (%)")
plt.tight_layout()
plt.show()


## 6. Key Exploratory Findings

The exploratory analysis provides the basis for the later inventory-risk,
ABC, safety-stock, reorder-point, supplier, and forecasting analyses.

Key observations:

1. Demand is concentrated among a relatively small group of products.
2. Packaging and safety-related categories represent substantial demand volume.
3. Warehouse demand is unevenly distributed.
4. Stockout exposure differs by warehouse.
5. Demand varies materially by month.
6. The November demand spike for Packaging Box is particularly important
   for forecasting and inventory planning.


## Conclusion

EDA identifies where inventory and demand-planning attention should be
concentrated. The next analytical stages evaluate inventory value, ABC
classification, turnover, demand variability, safety stock, reorder points,
EOQ, supplier reliability, and forecasting performance.
